In [1]:
"""
PIPELINE STAGE: Population Data Parsing, Structural Reconstruction & Spatial Alignment

INPUT: raw_data/population/population.pdf
OUTPUT: processed_data/steps/08_Population.xlsx
LIBRARIES: pandas, os, pdfplumber

1. OBJECTIVE:
    Parse a non-standard, flat-text population dataset and reconstruct it into a
    structured, tabular format. The pipeline extracts yearly population values
    for Turkish provinces, assigns spatial identifiers, removes duplicates,
    and produces a clean dataset compatible with energy, demographic, and
    socio-economic integration workflows.

2. RAW DATA STRUCTURE UNDERSTANDING:
    A. Input Format Characteristics:
       - The dataset is not in standard CSV/Excel format.
       - It is a single-line or flat-text file containing comma-separated tokens.
       - Data includes mixed elements:
           * Year identifiers
           * Province–plate combinations
           * Population values (scientific notation supported)

    B. Key Structural Pattern:
       Data follows sequential logic:

           Year → Province-Plate → Population
           
3. DATA INGESTION:
    A. File Reading:
       - Read entire file as a single string.
       - Use encoding:

           latin-1

    B. Tokenization:
       - Split raw content by comma delimiter.
       - Strip whitespace from each token.
       - Remove empty tokens.

    C. Result:
       A sequential token stream representing mixed structured data.

4. TEMPORAL STATE MANAGEMENT:
    A. Year Tracking Mechanism:
       Maintain a persistent variable:

           current_year

    B. Year Identification Rules:
       - Tokens that are numeric and within range:

           2021–2025

       are interpreted as year markers.

    C. State Propagation:
       Once a year is detected:
           - Assign it to current_year
           - Apply it to all subsequent records until replaced

5. RECORD PARSING LOGIC:
    A. Province–Plate Detection:
       Identify tokens containing:

           "-"

       indicating:

           Province-Plate structure

    B. Field Extraction:
       Split token using:

           rsplit("-", 1)

       to obtain:

           - Province name
           - Plate code

    C. Population Extraction:
       - Population value is located in the next token (i + 1)
       - Supports:
           * Standard integers
           * Floating values
           * Scientific notation (e.g., 1.58409E7)

    D. Data Construction:
       For each valid record create:

           {
             Year,
             Province,
             Plate,
             Population
           }

6. ERROR HANDLING & ROBUSTNESS:
    A. Boundary Protection:
       Ensure index safety when accessing next token (i + 1).

    B. Exception Handling:
       - Ignore malformed records
       - Continue processing without interruption

    C. Data Integrity:
       Skip invalid or incomplete token sequences silently.

7. DATAFRAME CONSTRUCTION:
    A. Conversion:
       Transform parsed records into a pandas DataFrame.

    B. Deduplication:
       Remove duplicate entries using:

           subset = ['Yıl', 'Plaka']
           keep = 'last'

    C. Purpose:
       Ensure one population value per province-year combination.

8. SPATIAL STANDARDIZATION:
    A. Geographic Keys:
       Use:

           Province (İl)
           Plate (Plaka)

       as primary spatial identifiers.

    B. Consistency:
       Align dataset structure with energy datasets for future merging.

9. OUTPUT GENERATION:
    A. Export File:

           processed_data/steps/08_Population.xlsx

    B. Export Format:
       - Excel (.xlsx)
       - No index column

10. PROCESS LOGGING & MONITORING:
    A. Runtime Messages:
       Display:
           - Completion confirmation
           - Total number of records processed

    B. Optional Debugging Insight:
       Provide total extracted row count for validation.

11. EXPECTED OUTPUT DATASET:
    The resulting dataset contains:

        - Year (2021–2025)
        - Province name (İl)
        - Official Plate code (Plaka)
        - Population (numeric, cleaned)

    The dataset is structured for:
        - Demographic integration with energy systems
        - Per capita energy analysis
        - Socio-economic modeling
        - Spatial-temporal machine learning applications

"""


import pandas as pd
import os



# Yollar
base_dir = r"C:\Users\w11\dergi2"
pop_path = os.path.join(base_dir, "raw_data", "population", "population.csv")
output_path = os.path.join(base_dir, "processed_data", "steps", "08_population.xlsx")

# 1. Dosyanın tamamını tek bir metin olarak oku
with open(pop_path, 'r', encoding='latin-1') as f:
    content = f.read()

# 2. Virgülle ayırıp bir liste yap
tokens = [t.strip() for t in content.split(',')]

data_list = []
current_year = None

# 3. Listeyi TEK TEK İNDEX İLE TARA (Hatanın çözüldüğü yer)
for i in range(len(tokens)):
    token = tokens[i]
    if not token: continue
    
    # Eğer token bir yıl ise (2020-2024 arası), onu 'current_year' yap
    if token.isdigit() and 2021 <= int(token) <= 2025:
        current_year = int(token)
        continue
    
    # Eğer içinde '-' varsa bu 'İl-Plaka' bilgisidir
    if '-' in token and current_year:
        try:
            if i + 1 < len(tokens):
    
                nufus_token = tokens[i + 1]
    
                # İl ve plaka ayır
                il_adi, plaka = token.rsplit('-', 1)
    
                # 2263373.0 ve 1.58409E7 gibi değerleri destekler
                nufus = int(float(nufus_token))
    
                data_list.append({
                    'Yıl': current_year,
                    'İl': il_adi,
                    'Plaka': int(plaka),
                    'Nüfus': nufus
                })
    
        except:
            continue
# 4. DataFrame ve Tekilleştirme
df = pd.DataFrame(data_list)
df_final = df.drop_duplicates(subset=['Yıl', 'Plaka'], keep='last')

# 5. Kaydet
df_final.to_excel(output_path, index=False)

print(f"Bitti! 2021-2025 arası tüm veriler alındı.")
print(f"Toplam kayıt: {len(df_final)}")

Bitti! 2021-2025 arası tüm veriler alındı.
Toplam kayıt: 405


In [4]:
import pandas as pd
import os
import pdfplumber

# =========================
# DOSYA YOLLARI
# =========================
base_dir = r"C:\Users\w11\dergi2"

pdf_path = os.path.join(base_dir, "raw_data", "population", "population.pdf")
output_path = os.path.join(base_dir, "processed_data", "steps", "088_population.xlsx")

# =========================
# 1. PDF OKU
# =========================
rows = []

with pdfplumber.open(pdf_path) as pdf:
    for page in pdf.pages:
        text = page.extract_text()

        if not text:
            continue

        lines = text.split("\n")

        for line in lines:
            parts = line.split()

            # header satırını atla
            if len(parts) < 4:
                continue
            if parts[0].lower() == "year":
                continue

            try:
                year = int(parts[0])
                population = float(parts[-1])
                plate = int(parts[-2])
                province = " ".join(parts[1:-2])

                rows.append([year, province, plate, population])

            except:
                continue


# =========================
# 2. DATAFRAME
# =========================
df = pd.DataFrame(rows, columns=["Yıl", "İl", "Plaka", "Nüfus"])

# --- TÜRKÇE BÜYÜK HARF DÖNÜŞÜMÜ ---
def turkce_upper(text):
    if pd.isna(text):
        return text
    text = str(text).strip()
    return text.replace("i", "İ").replace("ı", "I").upper()

df["İl"] = df["İl"].apply(turkce_upper)

# tekrarları temizle
df = df.drop_duplicates(subset=["Yıl", "Plaka"], keep="last")



# =========================
# 3. KAYDET
# =========================
os.makedirs(os.path.dirname(output_path), exist_ok=True)
df.to_excel(output_path, index=False)

print("Bitti! PDF başarıyla işlendi.")
print("Satır sayısı:", len(df))
print("Dosya:", output_path)

Bitti! PDF başarıyla işlendi.
Satır sayısı: 405
Dosya: C:\Users\w11\dergi2\processed_data\steps\088_population.xlsx
